# 44 — Análise qualitativa de erros

Orquestrador para a metodologia de análise qualitativa de erros descrita em `src/economy_classifier/error_analysis.py`. Toda lógica reusável vive lá; aqui só configuramos, amostramos e exportamos.

## Objetivo

Responder, com evidência manual, três perguntas que métricas agregadas não respondem:

1. **Construct validity (binário e multiclasse).** Que fração dos erros é "erro do modelo" vs. limite da rotulagem editorial? Reportável como `adjusted_correctness` na dissertação.
2. **Heterogeneidade da classe `colunas`.** As colunas que erram caem nas células densas previstas (colunas → mercado, colunas → poder, colunas → ilustrada) ou se distribuem aleatoriamente?
3. **Erro idiossincrático por família.** Quando TF-IDF, BERT e LLM discordam, quem erra de forma isolada — e o que esse erro revela sobre a família?

## Pipeline

```
predictions.csv + test.parquet
    -> load_predictions_with_text()
    -> build_{binary,multiclass,disagreement}_error_pool()
    -> summarize_errors_by_*()        # quantitativo, antes de anotar
    -> stratified_error_sample(seed=2026)
    -> export_annotation_template()    # CSV editável em planilha
    -> [anotação manual]
    -> load_annotated_sample() + summarize_annotations()
```

## Schema de anotação (controlled vocabulary)

Cada erro recebe **uma** das quatro categorias em `tipo_erro_anotado`:

| valor | quando aplicar |
|---|---|
| `rotulagem_editorial` | Texto é tematicamente da classe A, mas a editoria colocou em B. **Erro do rótulo, não do modelo.** |
| `tema_misto` | Texto cruza fronteiras temáticas (ex: política econômica). Ambos rótulos são defensáveis. |
| `modelo_erra` | Texto é claramente da classe A; o modelo previu B sem justificativa textual. **Erro do modelo.** |
| `ambiguo` | Texto não tem tema claro o suficiente para ser classificado por humano. |

Colunas auxiliares: `subtema_real` (texto livre), `editorialmente_economia` (bool), `signal_palavras` (termos diagnósticos), `notas` (livre).

## Tamanhos amostrais (calibrados pela banca)

- **Binário por modelo**: 50 FP + 50 FN = 100 anotações.
- **Multiclasse com foco em `colunas`**: ~50 anotações (estratificado por par true→pred).
- **Disagreement triangular** (TF-IDF/BERT/LLM): 30 anotações.
- Por família: 1 representante → ~280 anotações totais. ~2-3 dias de trabalho do aluno.

**Esta é a metodologia.** A execução real ocorre depois que os modelos forem reexecutados; este notebook foi projetado para ser rerodado sem mudança quando os novos `predictions.csv` chegarem.

## 0. Bootstrap (Colab + local)

Mesma lógica do notebook 41.

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), (
            f"Falta colab_splits.zip em {DRIVE_BASE}. "
            "Rode scripts/colab_pack.py local e suba o zip."
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")
        print("Splits extraidos em", SPLITS_DIR)

    RUNS_DIR = DRIVE_BASE / "runs"
    OUT_DIR = DRIVE_BASE / "error_analysis"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_DIR = REPO_DIR / "artifacts" / "runs"
    OUT_DIR = REPO_DIR / "artifacts" / "error_analysis"

OUT_DIR.mkdir(parents=True, exist_ok=True)
assert RUNS_DIR.exists(), f"RUNS_DIR não encontrado: {RUNS_DIR}"
assert SPLITS_DIR.exists(), f"SPLITS_DIR não encontrado: {SPLITS_DIR}"
print("RUNS_DIR:", RUNS_DIR)
print("SPLITS_DIR:", SPLITS_DIR)
print("OUT_DIR:", OUT_DIR)

## 1. Configuração

Escolha das runs a analisar. Edite as listas abaixo quando os modelos forem reexecutados — nada mais precisa mudar no notebook.

In [ ]:
import pandas as pd

from economy_classifier.error_analysis import (
    DEFAULT_SEED,
    build_binary_error_pool,
    build_disagreement_pool,
    build_multiclass_error_pool,
    detect_task,
    export_annotation_template,
    load_predictions_with_text,
    stratified_error_sample,
    summarize_errors_by_category,
    summarize_errors_by_confidence,
    summarize_errors_by_date,
    summarize_errors_by_text_length,
)

# Um representante por família para a análise binária.
BINARY_RUNS = {
    "tfidf_logreg": "tfidf_logreg_binary_test_set",
    "bert_bertimbau": "bert_bertimbau_binary_test_set",
    "llm_qwen_zero_shot": "llm_qwen2_5_7b_instruct_binary_zero_shot_test_set",
}

# Foco multiclasse: o melhor BERT (a tabela final indicará qual). Editar quando souber.
MULTICLASS_RUN = "bert_bertimbau_multiclass_test_set"

# Tamanhos de amostra (defensáveis perante a banca).
N_PER_STRATUM_BINARY = 50         # 50 FP + 50 FN por modelo binário
N_PER_STRATUM_MULTICLASS = 10     # ~10 por par true->pred mais denso
N_PER_STRATUM_DISAGREEMENT = 10   # ~10 por padrão de disagreement

# Foco multiclasse — classes editorialmente complicadas (FR-16 da revisão).
FOCUS_CLASSES = {"colunas", "mercado", "outros"}

print("DEFAULT_SEED:", DEFAULT_SEED)
print("binary runs:", list(BINARY_RUNS))
print("multiclass run:", MULTICLASS_RUN)

## 2. Carregar o conjunto de teste

O test set fixo (índices em `test_indices.csv`) é a base para todos os pools. Carregamos com metadados editoriais (`category`, `subcategory`, `date`, `link`).

In [ ]:
test_df = pd.read_parquet(SPLITS_DIR / "test.parquet")
print("test shape:", test_df.shape)
print("colunas:", list(test_df.columns))
print("label binário:", test_df['label'].value_counts().to_dict())
print("label multiclasse:")
print(test_df['label_multi'].value_counts().to_dict())

## 3. Pools binários (FP/FN) + agregados pré-anotação

Para cada run binária: junta texto + metadados, monta o pool de erros, e gera quatro tabelas diagnósticas que respondem **antes** de anotar:

- por `category` editorial real → onde o modelo confunde com `mercado`?
- por `subcategory` → granularidade que o modelo provavelmente não viu;
- por bin de `y_score` → erros de alta confiança são sinal de viés sistemático;
- por ano (`date`) → detectar drift.

Salva os agregados em `OUT_DIR/binary/<run>/` para reutilização nas células de agregação final do notebook 41 (seções 13–14).

In [ ]:
binary_pools = {}
binary_summaries = {}

for label, run_name in BINARY_RUNS.items():
    preds_path = RUNS_DIR / run_name / "predictions.csv"
    if not preds_path.exists():
        print(f"[skip] {label}: {preds_path} não existe ainda")
        continue

    joined = load_predictions_with_text(preds_path, test_df)
    if detect_task(joined) != "binary":
        print(f"[skip] {label}: predictions não são binárias")
        continue

    pool = build_binary_error_pool(joined)
    binary_pools[label] = pool

    out_dir = OUT_DIR / "binary" / label
    out_dir.mkdir(parents=True, exist_ok=True)

    summaries = {
        "by_category": summarize_errors_by_category(pool, column="category"),
        "by_subcategory": summarize_errors_by_category(pool, column="subcategory"),
        "by_error_type": summarize_errors_by_category(pool, column="error_type"),
        "by_confidence": summarize_errors_by_confidence(pool, n_bins=5),
        "by_text_length": summarize_errors_by_text_length(pool, n_bins=5),
        "by_date": summarize_errors_by_date(pool, freq="YS"),
    }
    for name, df in summaries.items():
        df.to_csv(out_dir / f"{name}.csv", index=False)
    binary_summaries[label] = summaries

    print(f"\n=== {label}: {len(pool)} erros ===")
    print("FP/FN:")
    print(summaries["by_error_type"].to_string(index=False))
    print("\nTop categorias com erro:")
    print(summaries["by_category"].head(8).to_string(index=False))
    print("\nDistribuição de confiança nos erros:")
    print(summaries["by_confidence"].to_string(index=False))

## 4. Amostragem estratificada e templates de anotação (binário)

Estratificação primária: `error_type` (FP vs FN). 50 por estrato → 100 anotações por modelo. O template salva as primeiras 2000 chars do texto + as colunas em branco do schema.

**Reprodutibilidade**: o seed é o `DEFAULT_SEED=2026`. Rodar este célula duas vezes produz exatamente a mesma amostra.

In [ ]:
binary_templates = {}
for label, pool in binary_pools.items():
    sample = stratified_error_sample(
        pool,
        n_per_stratum=N_PER_STRATUM_BINARY,
        stratify_by="error_type",
        seed=DEFAULT_SEED,
    )
    template_path = OUT_DIR / "binary" / label / "annotation_template.csv"
    export_annotation_template(sample, template_path)
    binary_templates[label] = template_path
    print(f"{label}: amostra={len(sample)}  ->  {template_path.relative_to(OUT_DIR.parent.parent) if template_path.is_relative_to(OUT_DIR.parent.parent) else template_path}")

## 5. Pool multiclasse (foco em `colunas`/`mercado`/`outros`)

Para a classe `colunas`, especialmente: a hipótese da metodologia é que os erros se concentram em pares previsíveis (`colunas → poder` quando a coluna é política, `colunas → mercado` quando é econômica, etc.). Se sim, F1 baixo em `colunas` é evidência da heterogeneidade editorial — não falha do modelo (FR-16 da revisão crítica).

Estratificação aqui é por `error_type` (= `"true->pred"`); 10 por par dá ~50 anotações. Pares com menos de 10 erros contribuem todos.

In [ ]:
preds_path = RUNS_DIR / MULTICLASS_RUN / "predictions.csv"
if preds_path.exists():
    joined_mc = load_predictions_with_text(preds_path, test_df)
    assert detect_task(joined_mc) == "multiclass", "predictions não são multiclasse"
    multi_pool = build_multiclass_error_pool(joined_mc, focus_classes=FOCUS_CLASSES)
    print(f"erros (foco={FOCUS_CLASSES}): {len(multi_pool)}")
    print("\nTop pares true->pred:")
    print(
        summarize_errors_by_category(multi_pool, column="error_type")
        .head(15).to_string(index=False)
    )

    out_dir = OUT_DIR / "multiclass" / MULTICLASS_RUN
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, fn in [
        ("by_error_type", lambda p: summarize_errors_by_category(p, column="error_type")),
        ("by_category", lambda p: summarize_errors_by_category(p, column="category")),
        ("by_subcategory", lambda p: summarize_errors_by_category(p, column="subcategory")),
        ("by_text_length", lambda p: summarize_errors_by_text_length(p, n_bins=5)),
        ("by_date", lambda p: summarize_errors_by_date(p, freq="YS")),
    ]:
        fn(multi_pool).to_csv(out_dir / f"{name}.csv", index=False)

    multi_sample = stratified_error_sample(
        multi_pool,
        n_per_stratum=N_PER_STRATUM_MULTICLASS,
        stratify_by="error_type",
        seed=DEFAULT_SEED,
    )
    multi_template = export_annotation_template(
        multi_sample, out_dir / "annotation_template.csv",
    )
    print(f"\namostra multiclasse: {len(multi_sample)}")
    print(f"template: {multi_template}")
else:
    print(f"[skip] {preds_path} não existe ainda — rode quando os modelos multiclasse estiverem prontos")
    multi_pool = None

## 6. Pool de disagreement entre famílias

Quando TF-IDF, BERT e LLM dão respostas diferentes para o mesmo texto: quem está certo, e quem erra de forma idiossincrática? Quatro padrões:

- `all_wrong` — exemplo "duro". Provável problema de rotulagem ou tema genuinamente ambíguo.
- `majority_wrong_one_right` — uma família vê algo que as outras não. Útil para entender força   de cada família (e pode embasar a discussão sobre quando usar ensemble).
- `majority_right_one_wrong` — uma família erra de forma isolada. Diagnóstico do viés daquela família.
- `split` — empate. Caso interessante para ensemble.

Estratificamos por padrão e amostramos 10 de cada → 30-40 anotações cross-família.

In [ ]:
available_predictions = {}
for label, run_name in BINARY_RUNS.items():
    preds_path = RUNS_DIR / run_name / "predictions.csv"
    if preds_path.exists():
        available_predictions[label] = pd.read_csv(preds_path)

if len(available_predictions) >= 2:
    disagreement = build_disagreement_pool(available_predictions, test_df)
    print(f"casos com disagreement: {len(disagreement)}")
    print("\npor padrão:")
    print(disagreement["disagreement_pattern"].value_counts().to_string())

    out_dir = OUT_DIR / "disagreement"
    out_dir.mkdir(parents=True, exist_ok=True)
    disagreement.to_csv(out_dir / "full_pool.csv", index=False)

    # Reaproveita stratified_error_sample tratando 'disagreement_pattern' como o estrato.
    # Para o template, copiamos colunas-padrão; method = 'disagreement_triangulation'
    disagreement["method"] = "disagreement_triangulation"
    disagreement["error_type"] = disagreement["disagreement_pattern"]
    disagreement_sample = stratified_error_sample(
        disagreement,
        n_per_stratum=N_PER_STRATUM_DISAGREEMENT,
        stratify_by="disagreement_pattern",
        seed=DEFAULT_SEED,
    )
    template = export_annotation_template(
        disagreement_sample, out_dir / "annotation_template.csv",
    )
    print(f"\namostra disagreement: {len(disagreement_sample)}")
    print(f"template: {template}")
else:
    print("[skip] disagreement precisa >=2 famílias com predictions disponíveis")

## 7. Guia de anotação (para o aluno/anotadores)

Abrir cada `annotation_template.csv` em LibreOffice/Excel/Sheets e preencher:

1. **`subtema_real`** — em uma frase: qual é o tema linguístico real do texto? Ex:    "polêmica em coluna sobre política monetária", "greve em multinacional", "crítica de filme".
2. **`tipo_erro_anotado`** — escolher exatamente um valor:
   - `rotulagem_editorial` — o texto **fala de** um tema mas a **editoria colocou em outra seção**      (ex: matéria sobre BNDES classificada como `cotidiano` porque caiu no caderno de SP).
   - `tema_misto` — o texto **legitimamente cruza temas** (ex: "reforma tributária aprovada pelo      Congresso" é simultaneamente `poder` e `mercado`).
   - `modelo_erra` — o texto é **claramente** de uma classe, e o modelo errou. Sem desculpa.
   - `ambiguo` — o texto é **curto/inespecífico demais** para classificar com confiança.
3. **`editorialmente_economia`** — `True` se o texto fala de economia/mercado tematicamente,    independente da editoria; `False` caso contrário. Útil para isolar a fração do corpus que é    "economia mal-rotulada".
4. **`signal_palavras`** — 3-6 termos que sinalizam o tema (ex: "Selic", "BNDES", "lucro líquido").    Útil para discutir features que TF-IDF capta vs. as que ele perde.
5. **`notas`** — qualquer observação livre (ex: "texto truncado, abrir link para confirmar").

**Critério de qualidade**: se duas pessoas anotam a mesma amostra independentemente, deve haver concordância >= 80% em `tipo_erro_anotado`. Para a dissertação, basta um anotador, mas reportar essa limitação.

## 8. Releitura e agregação pós-anotação

Esta seção roda depois que os CSVs forem anotados. Valida o vocabulário controlado, agrega as estatísticas-chave, e produz **`adjusted_correctness`** — o número que entra na seção de construct validity da dissertação.

Editar a lista de `ANNOTATED_FILES` quando os CSVs anotados estiverem disponíveis.

In [ ]:
import json

from economy_classifier.error_analysis import (
    load_annotated_sample,
    summarize_annotations,
)

# Adicione paths para CSVs já anotados. Vazio = pula a agregação.
ANNOTATED_FILES: dict[str, Path] = {
    # "tfidf_logreg_binary": OUT_DIR / "binary" / "tfidf_logreg" / "annotation_template_DONE.csv",
    # "bert_bertimbau_binary": OUT_DIR / "binary" / "bert_bertimbau" / "annotation_template_DONE.csv",
}

summaries_post: dict[str, dict] = {}
for label, path in ANNOTATED_FILES.items():
    if not path.exists():
        print(f"[skip] {label}: {path} não existe")
        continue
    annotated = load_annotated_sample(path, require_complete=True)
    summary = summarize_annotations(annotated)
    summaries_post[label] = summary
    print(f"\n=== {label} ===")
    print(f"  n={summary['n_annotated']}  adjusted_correctness={summary['adjusted_correctness']}")
    print("  tipos:")
    for t, info in summary["by_type"].items():
        print(f"    {t:>22s}: n={info['n']:>3d}  share={info['share']:.2%}")

if summaries_post:
    out = OUT_DIR / "annotation_summaries.json"
    out.write_text(json.dumps(summaries_post, indent=2, ensure_ascii=False))
    print(f"\nresumos salvos em: {out}")

## 9. Resumo da execução

Saídas esperadas em `OUT_DIR` (= `artifacts/error_analysis/` localmente, `My Drive/economy-classifier/error_analysis/` no Colab):

```
binary/
  <run_label>/
    by_category.csv          # erros por editoria real
    by_subcategory.csv
    by_error_type.csv        # FP/FN counts
    by_confidence.csv        # erros por bin de y_score
    by_text_length.csv
    by_date.csv
    annotation_template.csv  # CSV para anotar manualmente
multiclass/
  <run_name>/
    [mesmas tabelas, com error_type = 'true->pred']
    annotation_template.csv
disagreement/
  full_pool.csv
  annotation_template.csv
annotation_summaries.json    # após anotação manual
```

Os CSVs `by_*.csv` já são reportáveis na dissertação (sem precisar anotar). Os `annotation_template.csv` viram `annotation_template_DONE.csv` quando o aluno terminar de anotar — então a seção 8 deste notebook reagrega e produz `annotation_summaries.json`, que vai para a tabela final do artigo.